In [1]:
import pandas as pd
from pathlib import Path
from sklearn.linear_model import Ridge, PoissonRegressor
from sklearn.impute import SimpleImputer

from utility_functions import (
    FEATURE_COLS, CORE_SEASONS, TUNE_SEASONS, TRAIN_SEASONS, VAL_SEASONS,
    composite_loss, load_data, score,
)
from kalman_features import build_kalman_features

CURRENT_DIR = Path.cwd()
train, holdout = load_data(CURRENT_DIR)
baseline = pd.read_csv(CURRENT_DIR / 'baseline_validation_predictions.csv')

print(f'train  : {len(train)//2:,} matches  seasons {train.season.min()}-{train.season.max()}')
print(f'holdout: {len(holdout)//2:,} matches  dates {holdout.date.min().date()} to {holdout.date.max().date()}')


train  : 2,090 matches  seasons 2012-2022
holdout: 760 matches  dates 2023-08-11 to 2027-05-14


### Kalman filter features


In [2]:
# Holdout teams need the history built up during training.
# We run the KF on both datasets together, but record each team's strength
# BEFORE feeding in the match result -- so no future data leaks in.
combined = pd.concat([train, holdout], ignore_index=True)
combined = build_kalman_features(combined)

train_data   = combined[combined['match_id'].isin(train['match_id'])].copy()
holdout_data = combined[combined['match_id'].isin(holdout['match_id'])].copy()

cold_start_matches = train_data['kf_attack'].isna().sum() // 2
print(f'Cold-start matches (first 5 per team, no history yet): {cold_start_matches:,} -- imputed with training mean later')


Cold-start matches (first 5 per team, no history yet): 185 -- imputed with training mean later


In [3]:
SLIM_COLS = FEATURE_COLS + ['home', 'kf_attack']

# Use TRAIN seasons (2012-2019), skip cold-start rows with no KF history yet
train_ready = train_data[train_data['season'].isin(TRAIN_SEASONS)].dropna(subset=['kf_attack'])

corr = (train_ready[FEATURE_COLS + ['home', 'kf_attack', 'corners']]
        .corrwith(train_ready['corners'])
        .drop('corners')
        .sort_values(key=abs, ascending=False))
print('Correlation with corners (training set, post cold-start):')
print(corr.round(4).to_string())


Correlation with corners (training set, post cold-start):
feature_8     0.4110
kf_attack     0.2930
feature_7     0.2507
home          0.1640
feature_4     0.1221
feature_3     0.0701
feature_9     0.0669
feature_5    -0.0409
feature_1     0.0334
feature_6    -0.0156
feature_10   -0.0141
feature_2     0.0045


### Feature selection: kf_attack only


In [4]:
# kf_defense and opp_kf_* are collinear with feature_8 (r~0.3) -- they add noise, not signal. kf_attack alone wins.
val_data = train_data[train_data['season'].isin(VAL_SEASONS)].merge(baseline, on=['match_id', 'team'])
y_val    = val_data['corners'].to_numpy(float)
y_train  = train_ready['corners'].to_numpy(float)

KF_COLS = ['kf_attack', 'kf_defense', 'opp_kf_attack', 'opp_kf_defense']
ablation_groups = [
    ('no KF',         []),
    ('+ kf_attack',   ['kf_attack']),
    ('+ kf_atk+def',  ['kf_attack', 'kf_defense']),
    ('+ all KF',      KF_COLS),
]
imputer = SimpleImputer(strategy='mean')
for label, extra in ablation_groups:
    cols  = FEATURE_COLS + ['home'] + extra
    X_tr  = imputer.fit_transform(train_ready[cols])
    X_v   = imputer.transform(val_data[cols])
    preds = Ridge(alpha=10.0).fit(X_tr, y_train).predict(X_v)
    score(label, y_val, preds)


no KF                            MAE=2.2074  RMSE=2.7738  PoissonDev=1.2937
+ kf_attack                      MAE=2.1891  RMSE=2.7559  PoissonDev=1.2822
+ kf_atk+def                     MAE=2.1890  RMSE=2.7543  PoissonDev=1.2804
+ all KF                         MAE=2.1927  RMSE=2.7560  PoissonDev=1.2824


### Model and regularisation selection


In [5]:
# Fit on CORE(2012-17), pick alpha on TUNE(2018-19), touch VAL(2020-22) only at the end.
# Poisson beats Ridge: log-link prevents negative predictions, and its training loss matches how we evaluate count data.
core_data = train_data[train_data['season'].isin(CORE_SEASONS)].dropna(subset=['kf_attack'])
tune_data = train_data[train_data['season'].isin(TUNE_SEASONS)].dropna(subset=['kf_attack'])
y_core    = core_data['corners'].to_numpy(float)
y_tune    = tune_data['corners'].to_numpy(float)

imputer = SimpleImputer(strategy='mean')
X_core  = imputer.fit_transform(core_data[SLIM_COLS])
X_tune  = imputer.transform(tune_data[SLIM_COLS])

print('Ridge alpha sweep -- TUNE set')
ridge_scores = {}
for a in [1.0, 5.0, 10.0, 50.0, 100.0]:
    pred = Ridge(alpha=a).fit(X_core, y_core).predict(X_tune)
    score(f'  Ridge a={a}', y_tune, pred)
    ridge_scores[a] = composite_loss(y_tune, pred)
best_ridge_a = min(ridge_scores, key=ridge_scores.get)

print('Poisson alpha sweep -- TUNE set')
poisson_scores = {}
for a in [0.1, 0.5, 1.0, 2.0, 5.0]:
    pred = PoissonRegressor(alpha=a, max_iter=500).fit(X_core, y_core).predict(X_tune)
    score(f'  Poisson a={a}', y_tune, pred)
    poisson_scores[a] = composite_loss(y_tune, pred)
best_poisson_a = min(poisson_scores, key=poisson_scores.get)

print(f'Selected: Ridge a={best_ridge_a}  |  Poisson a={best_poisson_a}')


Ridge alpha sweep -- TUNE set
  Ridge a=1.0                    MAE=2.1545  RMSE=2.7144  PoissonDev=1.4080
  Ridge a=5.0                    MAE=2.1447  RMSE=2.6986  PoissonDev=1.3875
  Ridge a=10.0                   MAE=2.1380  RMSE=2.6860  PoissonDev=1.3718
  Ridge a=50.0                   MAE=2.1578  RMSE=2.6887  PoissonDev=1.3680
  Ridge a=100.0                  MAE=2.1934  RMSE=2.7310  PoissonDev=1.4060
Poisson alpha sweep -- TUNE set


  Poisson a=0.1                  MAE=2.1365  RMSE=2.6671  PoissonDev=1.3536
  Poisson a=0.5                  MAE=2.2210  RMSE=2.7723  PoissonDev=1.4480
  Poisson a=1.0                  MAE=2.2589  RMSE=2.8292  PoissonDev=1.5004
  Poisson a=2.0                  MAE=2.2881  RMSE=2.8732  PoissonDev=1.5425
  Poisson a=5.0                  MAE=2.3143  RMSE=2.9171  PoissonDev=1.5858
Selected: Ridge a=10.0  |  Poisson a=0.1


### KF hyperparameter search


In [6]:
# Q = how fast a team's strength can drift between matches; R = how much we trust a single match result.
# Smaller Q + larger R = slow, conservative estimates. Grid search on CORE->TUNE picks Q=0.03, R=30.0 (1st/56), confirmed by 4-fold CV.
Q_GRID = [0.03, 0.05, 0.10, 0.15, 0.30, 0.50, 1.00, 2.00]
R_GRID = [3.0,  5.0,  8.0,  11.0, 15.0, 20.0, 30.0]

results = []
for Q in Q_GRID:
    for R in R_GRID:
        data_with_kf = build_kalman_features(train, Q=Q, R=R)
        core_slice   = data_with_kf[data_with_kf['season'].isin(CORE_SEASONS)].dropna(subset=['kf_attack'])
        tune_slice   = data_with_kf[data_with_kf['season'].isin(TUNE_SEASONS)].dropna(subset=['kf_attack'])
        imputer = SimpleImputer(strategy='mean')
        pred = (PoissonRegressor(alpha=0.1, max_iter=500)
                .fit(imputer.fit_transform(core_slice[SLIM_COLS]), core_slice['corners'].to_numpy(float))
                .predict(imputer.transform(tune_slice[SLIM_COLS])))
        results.append({'Q': Q, 'R': R, 'composite': composite_loss(tune_slice['corners'].to_numpy(float), pred)})

results_df = pd.DataFrame(results).sort_values('composite').reset_index(drop=True)
best_Q = results_df.loc[0, 'Q']
best_R = results_df.loc[0, 'R']

print('Top 10 (Q, R) on TUNE set:')
print(results_df.head(10).round(4).to_string(index=False))
print(f'Selected: Q={best_Q}, R={best_R}')


Top 10 (Q, R) on TUNE set:
   Q    R  composite
0.03 30.0     1.8775
0.05 30.0     1.8782
0.03 20.0     1.8784
0.03 15.0     1.8791
0.05 20.0     1.8793
0.10 30.0     1.8795
0.03 11.0     1.8800
0.05 15.0     1.8802
0.15 30.0     1.8805
0.10 20.0     1.8809
Selected: Q=0.03, R=30.0


### Final evaluation (one-shot)


In [7]:
# Final check: fit on all training data (2012-2019), evaluate on VAL (2020-2022) exactly once.
all_data = pd.concat([train, holdout], ignore_index=True)
all_data = build_kalman_features(all_data, Q=best_Q, R=best_R)

train_data_final = all_data[all_data['match_id'].isin(train['match_id'])].copy()
train_rows       = train_data_final[train_data_final['season'].isin(TRAIN_SEASONS)].dropna(subset=['kf_attack'])
val_rows         = train_data_final[train_data_final['season'].isin(VAL_SEASONS)].merge(baseline, on=['match_id', 'team'])

imputer    = SimpleImputer(strategy='mean')
X_train    = imputer.fit_transform(train_rows[SLIM_COLS])
X_val      = imputer.transform(val_rows[SLIM_COLS])
y_train    = train_rows['corners'].to_numpy(float)
y_val      = val_rows['corners'].to_numpy(float)
y_baseline = val_rows['baseline_predicted_corners'].to_numpy(float)

ridge_model   = Ridge(alpha=best_ridge_a).fit(X_train, y_train)
poisson_model = PoissonRegressor(alpha=best_poisson_a, max_iter=500).fit(X_train, y_train)

score('Baseline',             y_val, y_baseline)
score('Ridge   + kf_attack',  y_val, ridge_model.predict(X_val))
score('Poisson + kf_attack',  y_val, poisson_model.predict(X_val))


Baseline                         MAE=2.2717  RMSE=2.8574  PoissonDev=1.3406
Ridge   + kf_attack              MAE=2.1891  RMSE=2.7559  PoissonDev=1.2822
Poisson + kf_attack              MAE=2.1602  RMSE=2.7140  PoissonDev=1.2423
